# M2 Gap — LSTM Baseline (Rolling-Window CV)

**Issue:** #49
**Owner:** Nelson
**Reviewer:** Mitchel
**Branch:** `artifact/m2-lstm-shap`

## Objective

Implement the LSTM neural-network baseline required by proposal §5.3 and compare it
against the shared Random Walk benchmark on the same terms as the ARIMA (#54) and
VAR (#28/#29) baselines.

## Methodology

- Target: `yield_spread_10y_2y` (first-differenced for modeling, evaluated in levels)
- Feature set: the 6-variable Gold-layer stationary set matching proposal §3.1/§5.3's
  committed predictors -- `d_yield_spread_10y_2y`, `d_overnight_rate`, `d_us_treasury_10y`,
  `d_fed_funds_rate`, `d_cpi_yoy`, `d_usdcad` -- built via `build_gold_features()` (#30)
  rather than re-deriving the merge/alignment logic a third time. `#28`/`#29`'s VAR
  baselines had converged on a 5-variable set that dropped `usdcad` (undocumented in
  `#28`; `#29` only matched `#28` for AIC-vs-BIC comparability, not because `usdcad` was
  found unnecessary), which contradicts §3.1's research question naming USD/CAD as a
  required transmission variable -- Aug 19 team decision: re-add it, starting here.
- Architecture: shallow single-layer LSTM, dropout regularization, early stopping on
  validation loss, per §5.3
- Rolling-window cross-validation: 5 sequential, expanding-window folds (not a static
  train/test split) with a chronological validation slice held out inside each fold's
  training block for early stopping
- Forecast horizons: 1, 5, and 20 trading days, evaluated in levels
  (`last_level + predicted_cumulative_change`), matching the ARIMA/VAR convention
- Benchmark: Random Walk (naive)
- Metrics: RMSE, MAE
- Diebold-Mariano test vs. naive, same methodology as #28/#29/#54 (Bartlett-kernel
  long-run variance, Harvey correction, and the forecast horizon passed explicitly as
  `h=h` -- omitting `h` silently falls back to the library's `h=1` default and
  understates the standard error at h=5/h=20, which is a live issue flagged on #54's
  PR #57)

## Why K sequential folds, not a per-origin refit

The ARIMA/VAR baselines refit at every 5-observation origin because those models fit
in milliseconds. Retraining a neural network that often is not tractable, so "rolling
CV" here means 5 sequential expanding-window folds -- each trains on all data up to
its test block and evaluates on the next block -- which still satisfies #49's
requirement of preserving temporal order and avoiding a single static split.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from dieboldmariano import dm_test

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from lstm_baseline import (  # noqa: E402
    load_common_sample,
    run_rolling_cv,
    train_final_model,
    save_final_model,
    HORIZONS,
    MIN_TRAIN,
    N_FOLDS,
    FEATURES,
    LEVEL_TARGET,
)

print("Project root:", PROJECT_ROOT)
print("Horizons:", HORIZONS)
print("Feature set:", FEATURES)


Project root: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5
Horizons: [1, 5, 20]
Feature set: ['d_yield_spread_10y_2y', 'd_overnight_rate', 'd_us_treasury_10y', 'd_fed_funds_rate', 'd_cpi_yoy', 'd_usdcad']


In [2]:
df = load_common_sample()

print("Common sample observations:", len(df))
print("Date range:", df["date"].min(), "->", df["date"].max())
df.head()


Common sample observations: 4268
Date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,d_yield_spread_10y_2y,d_overnight_rate,d_us_treasury_10y,d_fed_funds_rate,d_cpi_yoy,d_usdcad
0,2010-02-19,2.11,-0.01,0.0,-0.01,0.01,0.0,-0.0032
1,2010-02-22,2.12,0.01,0.0,0.02,-0.01,0.0,0.0008
2,2010-02-23,2.10,-0.02,0.0,-0.11,0.00,0.0,0.0089
3,2010-02-24,2.11,0.01,0.0,0.01,-0.01,0.0,0.0033
4,2010-02-25,2.12,0.01,0.0,-0.06,0.01,0.0,0.0125


In [3]:
# Rolling-window CV: 5 expanding-window folds, forecasts collected in levels
results_df, fold_diag_df = run_rolling_cv(df)

print("Total forecast rows:", len(results_df))
fold_diag_df


Total forecast rows: 11184


,fold,train_size,val_size,test_size,epochs_run,best_val_loss
0,0,425,75,745,12,0.611543
1,1,1059,186,745,20,0.778274
2,2,1692,298,745,12,0.955607
3,3,2325,410,745,14,0.981862
4,4,2958,522,748,9,2.285593


In [4]:
print("Missing forecasts:")
print(results_df[["lstm", "naive", "actual"]].isna().sum())

print("\nInfinite forecasts:")
print(np.isinf(results_df[["lstm", "naive", "actual"]]).sum())

print("\nForecast ranges:")
results_df[["actual", "naive", "lstm"]].agg(["min", "max"])


Missing forecasts:
lstm      0
naive     0
actual    0
dtype: int64

Infinite forecasts:
lstm      0
naive     0
actual    0
dtype: int64

Forecast ranges:


,actual,naive,lstm
min,-1.32,-1.32,-1.328933
max,1.64,1.64,1.638943


In [5]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]
    row = {"horizon": h}

    for name in ["lstm", "naive"]:
        row[f"{name}_rmse"] = np.sqrt(mean_squared_error(subset["actual"], subset[name]))
        row[f"{name}_mae"] = mean_absolute_error(subset["actual"], subset[name])

    row["lstm_rmse_improvement_pct"] = (row["naive_rmse"] - row["lstm_rmse"]) / row["naive_rmse"] * 100
    row["lstm_mae_improvement_pct"] = (row["naive_mae"] - row["lstm_mae"]) / row["naive_mae"] * 100

    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
metrics_df.round(6)


,horizon,lstm_rmse,lstm_mae,naive_rmse,naive_mae,lstm_rmse_improvement_pct,lstm_mae_improvement_pct
0,1,0.028858,0.020462,0.028824,0.020279,-0.116614,-0.900912
1,5,0.063211,0.047318,0.063069,0.047197,-0.225911,-0.256573
2,20,0.127351,0.096276,0.126128,0.095494,-0.969873,-0.819190


In [6]:
# Diebold-Mariano tests: LSTM vs Naive
# NOTE: `h=h` is passed explicitly on every call -- omitting it silently falls back
# to the library default of h=1 and understates the long-run variance at h=5/h=20
# (the bug flagged on PR #57 for the ARIMA baseline).

dm_results = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]
    actual = subset["actual"].to_numpy()
    naive = subset["naive"].to_numpy()
    lstm = subset["lstm"].to_numpy()

    dm_stat_sq, p_sq = dm_test(
        actual, lstm, naive,
        h=h, one_sided=False, harvey_correction=True, variance_estimator="bartlett",
    )
    dm_stat_abs, p_abs = dm_test(
        actual, lstm, naive,
        loss=lambda u, v: abs(u - v),
        h=h, one_sided=False, harvey_correction=True, variance_estimator="bartlett",
    )

    dm_results.append({
        "model": "lstm",
        "horizon_days": h,
        "dm_stat_squared_loss": dm_stat_sq,
        "dm_p_value_squared_loss": p_sq,
        "dm_stat_absolute_loss": dm_stat_abs,
        "dm_p_value_absolute_loss": p_abs,
    })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df.round(4)


,model,horizon_days,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss
0,lstm,1,1.4754,0.1402,7.6207,0.0000
1,lstm,5,0.7957,0.4263,0.7834,0.4335
2,lstm,20,1.1281,0.2593,0.7662,0.4436


In [7]:
# Train one final model on the full common sample (chronological train/val split,
# early stopping) for the SHAP interpretability notebook. This is separate from the
# rolling-CV folds above, which exist only to produce an honest out-of-sample
# forecast-accuracy evaluation.

final_model, scaling, X_full, origin_idx_full, final_diag = train_final_model(df)
print("Final model training diagnostics:", final_diag)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
save_final_model(final_model, scaling, OUTPUT_DIR / "lstm_final_model.pt")
print("Saved final model to", OUTPUT_DIR / "lstm_final_model.pt")


Final model training diagnostics: {'best_val_loss': 0.7031699419021606, 'epochs_run': 17}
Saved final model to /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/lstm_final_model.pt


In [8]:
metrics_df.to_csv(OUTPUT_DIR / "r3_lstm_vs_naive.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "r3_lstm_forecasts.csv", index=False)
dm_results_df.to_csv(OUTPUT_DIR / "r3_lstm_diebold_mariano.csv", index=False)
fold_diag_df.to_csv(OUTPUT_DIR / "r3_lstm_cv_folds.csv", index=False)

print("Saved LSTM outputs:")
print("- r3_lstm_vs_naive.csv")
print("- r3_lstm_forecasts.csv")
print("- r3_lstm_diebold_mariano.csv")
print("- r3_lstm_cv_folds.csv")
print("- lstm_final_model.pt")


Saved LSTM outputs:
- r3_lstm_vs_naive.csv
- r3_lstm_forecasts.csv
- r3_lstm_diebold_mariano.csv
- r3_lstm_cv_folds.csv
- lstm_final_model.pt


In [9]:
print("LSTM Baseline Conclusion")
print("=" * 60)
print(f"Common sample: {len(df)} observations, {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Rolling-window CV: {N_FOLDS} expanding-window folds, MIN_TRAIN={MIN_TRAIN}")
print()
for _, row in metrics_df.iterrows():
    h = int(row["horizon"])
    beats = "beats" if row["lstm_rmse"] < row["naive_rmse"] else "does not beat"
    print(f"h={h:>2}d  LSTM RMSE={row['lstm_rmse']:.6f}  Naive RMSE={row['naive_rmse']:.6f}  "
          f"({beats} naive, {row['lstm_rmse_improvement_pct']:+.2f}% RMSE)")
print()
for _, row in dm_results_df.iterrows():
    h = int(row["horizon_days"])
    sig_sq = "significant" if row["dm_p_value_squared_loss"] < 0.05 else "not significant"
    print(f"h={h:>2}d  DM (squared loss) p={row['dm_p_value_squared_loss']:.4f} ({sig_sq} at alpha=0.05)")


LSTM Baseline Conclusion
Common sample: 4268 observations, 2010-02-19 -> 2026-06-30
Rolling-window CV: 5 expanding-window folds, MIN_TRAIN=500

h= 1d  LSTM RMSE=0.028858  Naive RMSE=0.028824  (does not beat naive, -0.12% RMSE)
h= 5d  LSTM RMSE=0.063211  Naive RMSE=0.063069  (does not beat naive, -0.23% RMSE)
h=20d  LSTM RMSE=0.127351  Naive RMSE=0.126128  (does not beat naive, -0.97% RMSE)

h= 1d  DM (squared loss) p=0.1402 (not significant at alpha=0.05)
h= 5d  DM (squared loss) p=0.4263 (not significant at alpha=0.05)
h=20d  DM (squared loss) p=0.2593 (not significant at alpha=0.05)
